# Improving v2, judged on the 2026 holdout

The deployed model (lstm_raw v2) overfits in loss after about two epochs and misses most one-day floods. Earlier
retuning against 2024-2025 did not beat it, and those years have now been looked at several times, so this run
chooses everything on 2015-2025 and judges once on new data: 1 January to 30 September 2026, labelled from dated
reports before any model was scored on it (`data_collection/data/01_flood_labels/holdout_2026/LABELS_2026.md`).

**Files in `My Drive/flood_tuning/`**: `features_daily_audited.csv`, `features_holdout_2026.csv`, and the deployed
model in `v2_model/` (as for `v2_report.ipynb`). Run with **Runtime > Run all** on a T4 GPU in a fresh session,
about 1.5 hours.

**Plan, fixed before any result is seen**

1. Choosing data: leave-one-year-out over the labelled years 2015, 2016, 2018, 2019, 2021, 2022, 2023, 2024 and 2025
   (72 events, 205 flood days). Each year is scored by models trained on the other eight; eight seeds per fold.
2. Network: the v2 setting (32 units, dropout 0.3, a 14-day window, learning rate 0.001, batch 64, 15 epochs, event
   weights) with four regularisation settings: none (v2 as it is), label smoothing 0.1, L2 weight decay 0.0001, and
   both together; recurrent dropout 0.2 in the three regularised settings. Chosen by pooled out-of-fold average
   precision at epoch 15 (ties within 0.002 go to the simpler setting, in the order listed).
3. Threshold: best pooled out-of-fold day-level F1 (the rule v2 used, now on nine years), among score quantiles
   0.90 to 0.999, ties to the higher threshold.
4. Rainfall rule: an alarm is also raised when the three-day rainfall (CHIRPS) reaches R mm. R is chosen on the same
   out-of-fold days, with the LSTM threshold fixed, for the best day-level F1 (R from 20 to 120 mm in steps of 5).
   The rule is kept only if it raises out-of-fold day F1 by at least 0.01.
5. Calibration (for reporting only): a logistic fit of the out-of-fold scores to the labels, judged by the Brier
   score on 2026. It changes no alarm.
6. Final: the chosen network is trained on 2015-2025 and scored once on 2026, beside the deployed v2 at 0.528.
   **The new version replaces v2 only if its 2026 event F1 is higher than v2's and its 2026 day precision and day
   recall are both at least 0.4.** Otherwise v2 stays and the result is reported as tested.

In [ ]:
# 1. Setup
import os, json, time, zipfile
os.environ["TF_DETERMINISTIC_OPS"] = "1"
import numpy as np, pandas as pd, tensorflow as tf
tf.get_logger().setLevel("ERROR")
from numpy.lib.stride_tricks import sliding_window_view
print("TensorFlow", tf.__version__, "| Keras", tf.keras.__version__ if hasattr(tf.keras, "__version__") else "")
print("GPU:", tf.config.list_physical_devices("GPU") or "none (Runtime > Change runtime type > T4 GPU)")

In [ ]:
# 2. Data: upload features_daily_audited.csv from
#    data_collection/data/08_model_ready/ in the project folder, or point DATA_PATH at a copy in Google Drive.
from google.colab import drive, files
DATA_PATH = "/content/drive/MyDrive/flood_tuning/features_daily_audited.csv"   # optional Drive copy
OUT_DIR = "/content/drive/MyDrive/flood_tuning/improve_results"                       # results saved here if Drive is used
try:
    drive.mount("/content/drive")
except Exception as e:
    print("Drive not mounted:", e)
if not os.path.exists(DATA_PATH):
    print("Upload features_daily_audited.csv")
    up = files.upload()
    DATA_PATH = "/content/" + next(iter(up))
    OUT_DIR = "/content/improve_results"
os.makedirs(OUT_DIR, exist_ok=True)
RAW = pd.read_csv(DATA_PATH, parse_dates=["date"]).sort_values(["catchment", "date"])
HOLD_PATH = os.path.join(os.path.dirname(DATA_PATH), "features_holdout_2026.csv")
if not os.path.exists(HOLD_PATH):
    print("Upload features_holdout_2026.csv")
    up = files.upload(); HOLD_PATH = "/content/" + next(iter(up))
HOLD = pd.read_csv(HOLD_PATH, parse_dates=["date"], keep_default_na=False)
HOLD["event_id"] = HOLD["event_id"].replace("", np.nan)
RAW = pd.concat([RAW, HOLD[[c for c in HOLD.columns if c in RAW.columns]]], ignore_index=True)
RAW = RAW.sort_values(["catchment", "date"]).reset_index(drop=True)
RAW["rain_sum_3d"] = RAW.groupby("catchment")["rain_mm"].transform(lambda x: x.rolling(3, min_periods=1).sum())
print(len(RAW), "rows,", RAW["date"].min().date(), "to", RAW["date"].max().date())
print(RAW.groupby(RAW["date"].dt.year == 2026)["label_status"].value_counts().to_dict())

In [ ]:
# 3. Shared code, copied from the repository (sequences.py, save_final_model.py, event_train_evaluate.py)
SEQ_FEATS = ["rain_mm", "imerg_max_rate", "imerg_total_mm", "imerg_hours_ge5", "era5_rain_mm", "soil_1", "soil_2",
             "temp_c", "dewpoint_depression", "pressure_hpa", "month_sin", "month_cos"]
CATCHMENTS = ["ngong", "mathare", "nairobi"]
DEV_FOLDS = [(range(2015, 2019), [2019]), (range(2015, 2021), [2021, 2022]), (range(2015, 2023), [2023])]
FINAL_FOLD = (range(2015, 2024), [2024, 2025])
SEEDS = list(range(8))
DAY = pd.Timedelta(days=1)
QS = np.round(np.arange(0.90, 0.9991, 0.001), 3)
_windows = {}


def make_windows(t, drop=()):
    # one window per catchment-day: the t days ending on that day, labelled with the last day
    if (t, drop) not in _windows:
        df = RAW.copy()
        for c in CATCHMENTS:
            df[f"is_{c}"] = (df["catchment"] == c).astype("float32")
        feats = [f for f in SEQ_FEATS + [f"is_{c}" for c in CATCHMENTS] if f not in drop]
        xs, metas = [], []
        for c, g in df.groupby("catchment"):
            vals = g[feats].values.astype("float32")
            xs.append(sliding_window_view(vals, (t, vals.shape[1]))[:, 0])
            metas.append(g.iloc[t - 1:][["catchment", "date", "label_status", "event_id"]].copy())
        meta = pd.concat(metas).reset_index(drop=True)
        meta["year"] = meta["date"].dt.year
        _windows[(t, drop)] = (np.concatenate(xs), meta, feats)
    return _windows[(t, drop)]


REG = {"l2": 0.0, "smooth": 0.0, "rec_dropout": 0.0}  # regularisation, set per setting in cell 4


def make_model(n_feats, t, units, dropout, lr):
    l2 = tf.keras.regularizers.l2(REG["l2"]) if REG["l2"] else None
    m = tf.keras.Sequential([
        tf.keras.layers.Input((t, n_feats)),
        tf.keras.layers.LSTM(units, kernel_regularizer=l2, recurrent_regularizer=l2,
                             recurrent_dropout=REG["rec_dropout"]),
        tf.keras.layers.Dropout(dropout),
        tf.keras.layers.Dense(1, activation="sigmoid", kernel_regularizer=l2),
    ])
    m.compile(optimizer=tf.keras.optimizers.Adam(lr),
              loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=REG["smooth"]))
    return m


def fold_data(X, meta, train_years, val_years):
    st = meta["label_status"].values
    flood, known = st == "flood", np.isin(st, ["flood", "no_flood"])
    tr = meta["year"].isin(list(train_years)).values & known
    va = meta["year"].isin(list(val_years)).values
    flat = X[tr].reshape(-1, X.shape[2])
    mu, sd = flat.mean(0), flat.std(0) + 1e-6
    w = float(np.sqrt((tr & ~flood).sum() / (tr & flood).sum()))          # class weight
    eid = meta["event_id"].where(flood)
    L = meta[tr & flood].groupby(eid[tr & flood]).size()                   # days per event
    sw = np.where(flood, w * L.mean() / eid.map(L).fillna(1).values, 1.0)  # event weights
    return (X[tr] - mu) / sd, flood[tr].astype("float32"), sw[tr], (X[va] - mu) / sd, va, mu, sd


def run_setting(t, units, dropout, lr, max_ep, folds, drop=(), keep_models=False, log=None):
    # eight seeds; scores after every epoch, averaged over seeds -> {epoch: DataFrame(val, catchment, date, score)}
    X, meta, feats = make_windows(t, drop)
    out, models = {}, []
    for train_years, val_years in folds:
        Xtr, ytr, sw, Xva, va, mu, sd = fold_data(X, meta, train_years, val_years)
        S = np.zeros((max_ep, len(Xva)))
        for seed in SEEDS:
            tf.keras.utils.set_random_seed(seed)
            m = make_model(X.shape[2], t, units, dropout, lr)
            for ep in range(max_ep):
                h = m.fit(Xtr, ytr, sample_weight=sw, epochs=1, batch_size=64, verbose=0)
                S[ep] += m.predict(Xva, batch_size=1024, verbose=0).ravel() / len(SEEDS)
                if log is not None:
                    log.append({"val": "-".join(map(str, val_years)), "seed": seed, "epoch": ep + 1,
                                "loss": h.history["loss"][0]})
            if keep_models:
                models.append(m)
        base = meta.loc[va, ["catchment", "date"]].reset_index(drop=True)
        for ep in range(max_ep):
            d = base.copy(); d["val"] = "-".join(map(str, val_years)); d["score"] = S[ep]
            out.setdefault(ep + 1, []).append(d)
    out = {ep: pd.concat(v, ignore_index=True) for ep, v in out.items()}
    return (out, models, feats, mu, sd) if keep_models else out


LAB = RAW[["catchment", "date", "label_status", "event_id"]]


def prepare(d):
    m = d.merge(LAB, on=["catchment", "date"])
    parts = []
    for _, g in m.groupby("val"):
        ev = (g[g["label_status"] == "flood"].groupby("event_id")
              .agg(catchment=("catchment", "first"), start=("date", "min"), end=("date", "max"), days=("date", "size")))
        known = g[g["label_status"].isin(["flood", "no_flood"])]
        parts.append((g, ev, known["label_status"].eq("flood").values, known["score"].values))
    return parts


def ap(y, s):
    o = np.argsort(-s, kind="mergesort")
    y, s = y[o], s[o]
    last = np.r_[np.diff(s) != 0, True]
    tp = np.cumsum(y)[last]
    return float(np.sum(np.diff(np.r_[0, tp / tp[-1]]) * tp / np.arange(1, len(y) + 1)[last]))


def pooled_ap(parts):
    return ap(np.concatenate([p[2] for p in parts]), np.concatenate([p[3] for p in parts]))


def stats(parts, thr, cy):
    n = det = det1 = n1 = eps = false_eps = tp = fp = fn = 0
    for g, ev, y, s in parts:
        a = s >= thr
        tp += int((a & y).sum()); fp += int((a & ~y).sum()); fn += int((~a & y).sum())
        alarms = g[g["score"] >= thr]
        for e in ev.itertuples():
            x = alarms[alarms["catchment"] == e.catchment]
            hit = bool(((x["date"] >= e.start - DAY) & (x["date"] <= e.end + DAY)).any())
            det += hit
            if e.days == 1:
                n1 += 1; det1 += hit
        n += len(ev)
        for c, x in alarms.sort_values("date").groupby("catchment"):
            evc = ev[ev["catchment"] == c]
            for _, ep in x.groupby((x["date"].diff() != DAY).cumsum()):
                eps += 1
                false_eps += not ((evc["start"] - DAY <= ep["date"].max()) & (evc["end"] + DAY >= ep["date"].min())).any()
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn)
    er = det / n
    ep_p = (eps - false_eps) / eps if eps else 0.0
    return {"tp": tp, "fp": fp, "fn": fn, "precision": p, "recall": r, "f1": 2 * p * r / (p + r) if p + r else 0.0,
            "events": n, "detected": det, "one_day_detected": f"{det1}/{n1}", "multi_day_detected": f"{det - det1}/{n - n1}",
            "event_recall": er, "episodes": eps, "false_episodes": false_eps, "episode_precision": ep_p,
            "event_f1": 2 * ep_p * er / (ep_p + er) if ep_p + er else 0.0, "false_eps_per_cy": false_eps / cy}


def pick_threshold(dev_scores):
    dp = prepare(dev_scores)
    grid = np.unique(np.quantile(dev_scores["score"], QS))
    sw = pd.DataFrame([{"threshold": t, **stats(dp, t, 12)} for t in grid])
    return sw.sort_values(["f1", "threshold"], ascending=[False, False]).iloc[0]


def wilson(k, n, z=1.96):
    p = k / n; c = (p + z * z / (2 * n)) / (1 + z * z / n)
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return round(c - h, 2), round(c + h, 2)


def loss_history(t, units, dropout, lr, max_ep, fold, seeds, drop=()):
    # training and validation loss per epoch on one fold, averaged over seeds; both weighted the same way
    X, meta, _ = make_windows(t, drop)
    Xtr, ytr, sw, Xva, va, _, _ = fold_data(X, meta, *fold)
    st = meta["label_status"].values
    flood = st == "flood"
    tr_years = meta["year"].isin(list(fold[0])).values & np.isin(st, ["flood", "no_flood"])
    w = float(np.sqrt((tr_years & ~flood).sum() / (tr_years & flood).sum()))
    eid = meta["event_id"].where(flood)
    L = meta[va & flood].groupby(eid[va & flood]).size()
    known = np.isin(st[va], ["flood", "no_flood"])
    yva = flood[va].astype("float32")
    swva = np.where(yva == 1, w * L.mean() / eid[va].map(L).fillna(1).values, 1.0)
    hist = []
    for seed in seeds:
        tf.keras.utils.set_random_seed(seed)
        m = make_model(X.shape[2], t, units, dropout, lr)
        h = m.fit(Xtr, ytr, sample_weight=sw, epochs=max_ep, batch_size=64, verbose=0,
                  validation_data=(Xva[known], yva[known], swva[known]))
        hist.append([h.history["loss"], h.history["val_loss"]])
    return np.array(hist).mean(0)

print("ready")

In [ ]:
# 4. Leave-one-year-out scores over 2015-2025 for the four regularisation settings (about 15 minutes each)
YEARS = [2015, 2016, 2018, 2019, 2021, 2022, 2023, 2024, 2025]
LOYO = [([y2 for y2 in YEARS if y2 != y], [y]) for y in YEARS]
V2 = {"lr": 0.001, "window": 14, "units": 32, "dropout": 0.3}
EP = 15
CY = 3 * len(YEARS)
SETTINGS = [("v2 as deployed", {"l2": 0.0, "smooth": 0.0, "rec_dropout": 0.0}),
            ("label smoothing", {"l2": 0.0, "smooth": 0.1, "rec_dropout": 0.2}),
            ("weight decay", {"l2": 1e-4, "smooth": 0.0, "rec_dropout": 0.2}),
            ("both", {"l2": 1e-4, "smooth": 0.1, "rec_dropout": 0.2})]
rows, oof = [], {}
for name, reg in SETTINGS:
    f = f"{OUT_DIR}/oof_{name.replace(' ', '_')}.csv"
    if os.path.exists(f):
        s = pd.read_csv(f, parse_dates=["date"])
    else:
        REG.update(reg); t0 = time.time()
        s = run_setting(V2["window"], V2["units"], V2["dropout"], V2["lr"], EP, LOYO)[EP]
        s.to_csv(f, index=False); print(f"{name}: {time.time() - t0:.0f}s")
    oof[name] = s
    rows.append({"setting": name, **reg, "oof_ap": pooled_ap(prepare(s))})
search = pd.DataFrame(rows); search.to_csv(f"{OUT_DIR}/search.csv", index=False)
print(search.round(4).to_string(index=False))
best = search["oof_ap"].max()
CHOSEN = search[search["oof_ap"] >= best - 0.002].iloc[0]["setting"]
REG.update(dict(SETTINGS)[CHOSEN])
print("Chosen network:", CHOSEN, REG)

In [ ]:
# 5. Threshold, rainfall rule and calibration, all on the out-of-fold days
s = oof[CHOSEN].merge(RAW[["catchment", "date", "rain_sum_3d"]], on=["catchment", "date"])
thr_row = pick_threshold(s)
THR = float(thr_row["threshold"])
base = stats(prepare(s), THR, CY)
print(f"LSTM threshold {THR:.4f}: out-of-fold precision {base['precision']:.3f}, recall {base['recall']:.3f}, "
      f"F1 {base['f1']:.3f}, events {base['detected']}/{base['events']}")


def combined(df, thr, r):
    # one score that is above THR exactly when the LSTM or the rule alarms
    d = df.copy()
    d["score"] = np.where(d["rain_sum_3d"] >= r, np.maximum(d["score"], thr + 1e-6), d["score"])
    return d


rule_rows = []
for r in range(20, 125, 5):
    q = stats(prepare(combined(s, THR, r)), THR, CY)
    rule_rows.append({"R_mm": r, **{k: q[k] for k in ("precision", "recall", "f1", "detected", "event_f1")}})
rule = pd.DataFrame(rule_rows); rule.to_csv(f"{OUT_DIR}/rule_search.csv", index=False)
b = rule.sort_values(["f1", "R_mm"], ascending=[False, False]).iloc[0]
USE_RULE = bool(b["f1"] >= base["f1"] + 0.01)
R_MM = int(b["R_mm"]) if USE_RULE else None
print(rule.round(3).to_string(index=False))
print(f"Rule {'kept' if USE_RULE else 'not kept'}: best R {int(b['R_mm'])} mm gives out-of-fold F1 {b['f1']:.3f} "
      f"against {base['f1']:.3f} without it")

from sklearn.linear_model import LogisticRegression
known = s[s.merge(RAW[["catchment", "date", "label_status"]], on=["catchment", "date"])["label_status"]
          .isin(["flood", "no_flood"]).values]
lab = known.merge(RAW[["catchment", "date", "label_status"]], on=["catchment", "date"])["label_status"].eq("flood")
logit = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
CAL = LogisticRegression().fit(logit(known["score"].values).reshape(-1, 1), lab.values)
json.dump({"network": CHOSEN, "reg": REG, "threshold": THR, "use_rule": USE_RULE, "rule_mm": R_MM,
           "oof": {k: base[k] for k in ("precision", "recall", "f1", "detected", "events")},
           "calibration": [float(CAL.coef_[0][0]), float(CAL.intercept_[0])]},
          open(f"{OUT_DIR}/choice.json", "w"), indent=2, default=float)
print(json.load(open(f"{OUT_DIR}/choice.json")))

In [ ]:
# 6. Loss curves of the chosen network (2015-2022 -> 2023 and 2015-2024 -> 2025, 4 seeds), for the report
import matplotlib.pyplot as plt
curves = {"2015-2022 -> 2023": loss_history(14, 32, 0.3, 0.001, EP, ([y for y in YEARS if y <= 2022], [2023]), seeds=range(4)),
          "2015-2024 -> 2025": loss_history(14, 32, 0.3, 0.001, EP, ([y for y in YEARS if y <= 2024], [2025]), seeds=range(4))}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), dpi=150)
out = {"epoch": range(1, EP + 1)}
for ax, (title, h) in zip(axes, curves.items()):
    ax.plot(range(1, EP + 1), h[0], color="#2a78d6", lw=2, label="Training loss")
    ax.plot(range(1, EP + 1), h[1], color="#eb6834", lw=2, label="Validation loss")
    ax.set_title(f"{CHOSEN}: {title}", fontsize=10); ax.set_xlabel("Epoch"); ax.legend(frameon=False, fontsize=9)
    for sp in ["top", "right"]: ax.spines[sp].set_visible(False)
    out[f"train_{title}"] = h[0]; out[f"val_{title}"] = h[1]
axes[0].set_ylabel("Weighted cross-entropy")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/loss_curves.png"); plt.show()
pd.DataFrame(out).to_csv(f"{OUT_DIR}/loss_curves.csv", index=False)

## Final test on 2026: run this cell once

Trains the chosen network on 2015-2025, scores 1 January to 30 September 2026 once, and compares it with the
deployed v2. Do not change anything above after running it.

In [ ]:
# 7. FINAL TEST on 2026 (once)
ch = json.load(open(f"{OUT_DIR}/choice.json"))
HOLDOUT = (YEARS, [2026])
CY26 = 3 * 0.75  # three catchments, nine months


def report(s, thr, rule_mm=None):
    s = s.merge(RAW[["catchment", "date", "rain_sum_3d"]], on=["catchment", "date"])
    if rule_mm:
        s = combined(s, thr, rule_mm)
    parts = prepare(s)
    r = {"ap": pooled_ap(parts), **stats(parts, thr, CY26)}
    r["tn"] = int(sum(len(p[2]) for p in parts)) - r["tp"] - r["fp"] - r["fn"]
    r["precision_ci"] = wilson(r["tp"], r["tp"] + r["fp"]) if r["tp"] + r["fp"] else None
    r["recall_ci"] = wilson(r["tp"], r["tp"] + r["fn"]); r["event_recall_ci"] = wilson(r["detected"], r["events"])
    return r, s


# deployed v2, unchanged
MODEL_DIR = "/content/drive/MyDrive/flood_tuning/v2_model"
if not os.path.exists(f"{MODEL_DIR}/config.json"):
    os.makedirs("/content/v2_model", exist_ok=True)
    for name, data in files.upload().items():
        open(f"/content/v2_model/{name}", "wb").write(data)
    MODEL_DIR = "/content/v2_model"
cfg = json.load(open(f"{MODEL_DIR}/config.json"))
X, meta, feats = make_windows(cfg["window"])
X = X[:, :, [feats.index(f) for f in cfg["features"]]]
h26 = meta["year"].eq(2026).values
Xh = ((X[h26] - np.array(cfg["mean"])) / np.array(cfg["std"])).astype("float32")
REG.update({"l2": 0.0, "smooth": 0.0, "rec_dropout": 0.0})
sc = np.zeros(len(Xh))
for seed in cfg["seeds"]:
    m = make_model(X.shape[2], cfg["window"], 32, 0.3, 0.001)
    m.load_weights(f"{MODEL_DIR}/lstm_raw_seed{seed}.weights.h5")
    sc += m.predict(Xh, batch_size=1024, verbose=0).ravel() / len(cfg["seeds"])
v2s = meta.loc[h26, ["catchment", "date"]].reset_index(drop=True); v2s["val"] = "2026"; v2s["score"] = sc
res = {}
res["v2 deployed (0.528)"], _ = report(v2s, cfg["threshold"])

# the chosen network, trained on 2015-2025
REG.update(ch["reg"])
scores, models, feats_new, mu, sd = run_setting(14, 32, 0.3, 0.001, EP, [HOLDOUT], keep_models=True)
new = scores[EP]
res[f"{ch['network']} ({ch['threshold']:.3f})"], _ = report(new, ch["threshold"])
if ch["use_rule"]:
    res[f"{ch['network']} + rule {ch['rule_mm']} mm"], _ = report(new, ch["threshold"], ch["rule_mm"])
new.to_csv(f"{OUT_DIR}/final_scores_2026.csv", index=False); v2s.to_csv(f"{OUT_DIR}/v2_scores_2026.csv", index=False)

# calibration, reported only: Brier score on labelled 2026 days
k = new.merge(RAW[["catchment", "date", "label_status"]], on=["catchment", "date"])
k = k[k["label_status"].isin(["flood", "no_flood"])]
y26 = k["label_status"].eq("flood").values
a, b0 = ch["calibration"]
cal = 1 / (1 + np.exp(-(a * logit(k["score"].values) + b0)))
brier = {"raw": float(np.mean((k["score"].values - y26) ** 2)), "calibrated": float(np.mean((cal - y26) ** 2))}

table = pd.DataFrame(res)
print(table.to_string()); print("Brier score of the new network on 2026:", brier)
final_key = list(res)[-1]
f = res[final_key]; v = res["v2 deployed (0.528)"]
adopt = f["event_f1"] > v["event_f1"] and f["precision"] >= 0.4 and f["recall"] >= 0.4
print(); print(f"Decision rule: {final_key} {'REPLACES' if adopt else 'does NOT replace'} v2 "
      f"(event F1 {f['event_f1']:.3f} against {v['event_f1']:.3f}; precision {f['precision']:.3f}, recall {f['recall']:.3f})")
json.dump({"results": res, "brier": brier, "adopt": adopt, "final": final_key}, open(f"{OUT_DIR}/final_results.json", "w"),
          indent=2, default=lambda o: list(o) if isinstance(o, tuple) else float(o))

d = "/content/models/flood_lstm_v5"
os.makedirs(d, exist_ok=True)
for seed, m in zip(SEEDS, models):
    m.save_weights(f"{d}/lstm_raw_seed{seed}.weights.h5")
json.dump({"name": "lstm_raw", "version": "v5", "training": "event-weighted", "features": feats_new, "window": 14,
           "units": 32, "dropout": 0.3, "learning_rate": 0.001, "epochs": EP, "regularisation": ch["reg"],
           "mean": mu.tolist(), "std": sd.tolist(), "threshold": ch["threshold"],
           "rule_rain_3d_mm": ch["rule_mm"] if ch["use_rule"] else None, "calibration": ch["calibration"],
           "seeds": SEEDS, "train_years": [2015, 2025], "labels": "features_daily_audited.csv",
           "target": "today", "tuned_in": "colab/improve_v2.ipynb", "adopted": adopt},
          open(f"{d}/config.json", "w"), indent=2)
with zipfile.ZipFile("/content/improve_v2.zip", "w") as z:
    for root, _, fs in os.walk("/content/models"):
        for fn in fs:
            z.write(os.path.join(root, fn), os.path.relpath(os.path.join(root, fn), "/content"))
    for fn in os.listdir(OUT_DIR):
        if not fn.startswith("oof_"):
            z.write(f"{OUT_DIR}/{fn}", f"results/{fn}")
files.download("/content/improve_v2.zip")